In [1]:
# ============= 1) Imports & Global Options =============
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import re

from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from pathlib import Path

print("✓ Imports ready")

✓ Imports ready


In [2]:
# ============= 2) Preprocessing Utils =============
STOP_WORDS = {
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours','yourself','yourselves',
    'he','him','his','himself','she','her','hers','it','its','itself','they','them','their',
    'theirs','themselves','what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does','did','doing','a','an',
    'the','and','but','if','or','because','as','until','while','of','at','by','for','with','about',
    'against','between','into','through','during','before','after','above','below','to','from','up',
    'down','in','out','on','off','over','under','again','further','then','once','here','there','when',
    'where','why','how','all','any','both','each','few','more','most','other','some','such','no','nor',
    'not','only','own','same','so','than','too','very','s','t','can','will','just','don','should','now'
}
stemmer = PorterStemmer()

def preprocess_text(text: str) -> str:
    if not isinstance(text, str): return ""
    text = re.sub(r'[^a-z\s]', ' ', str(text).lower())
    tokens = [stemmer.stem(w) for w in text.split() if w not in STOP_WORDS]
    return " ".join(tokens)

def clean_text_for_excel(text: str) -> str:
    if isinstance(text, str):
        return re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', text)
    return text

print("✓ Preprocessing ready")

✓ Preprocessing ready


In [3]:
# ============= 3) Articles: Load Raw =============
ARTICLES_CSV = "Articles.csv"
articles = pd.read_csv(ARTICLES_CSV, header=None, low_memory=False)
cols = list(articles.columns)
cols[0], cols[1], cols[2] = "article_id", "title", "abstract_start"
articles.columns = cols
articles = articles.drop(index=0).reset_index(drop=True)
print(f"✓ Raw articles loaded: {articles.shape}")

✓ Raw articles loaded: (2261, 291)


In [4]:
# ============= 4) Articles: Merge Abstract Fragments =============
cols_after = articles.columns[articles.columns.get_loc('abstract_start') + 1:]
articles['abstract_start'] = (
    articles['abstract_start'].astype(str) + ' ' +
    articles.apply(lambda r: ' '.join(r[cols_after].dropna().astype(str).replace('nan','')), axis=1)
)
articles = articles.drop(columns=cols_after)
print("✓ Abstract fragments merged")

✓ Abstract fragments merged


In [5]:
# ============= 5) Articles: Build Final df & Save XLSX =============
split_cols = articles['article_id'].str.split(',', n=2, expand=True)
if split_cols.shape[1] == 2: split_cols[2] = np.nan
split_cols.columns = ['article_id_clean', 'title_clean', 'abstract_head']

articles['abstract'] = (split_cols['abstract_head'].fillna('') + ' ' + articles['abstract_start'].fillna(''))
articles['abstract'] = (articles['abstract'].str.replace(r'\s+',' ',regex=True)
                        .str.replace(r'^"|"$','',regex=True).str.strip())
articles['title'] = (split_cols['title_clean'].str.replace(r'\s+',' ',regex=True)
                     .str.replace(r'^"|"$','',regex=True).str.strip())
articles['article_id'] = pd.to_numeric(split_cols['article_id_clean'], errors='coerce')
articles = articles.dropna(subset=['article_id'])
articles['article_id'] = articles['article_id'].astype(int)

df = articles[['article_id','title','abstract']].copy()

df_excel = df.copy()
df_excel['title'] = df_excel['title'].apply(clean_text_for_excel)
df_excel['abstract'] = df_excel['abstract'].apply(clean_text_for_excel)
df_excel.to_excel("cleaned_articles.xlsx", index=False)

display(df.head(5).style.hide(axis='index'))
print(f"✓ Final df: {df.shape} | saved: cleaned_articles.xlsx")

article_id,title,abstract
4,comparative assessment of largescale data sets of proteinprotein interactions,Comprehensive protein protein interaction maps promise to reveal many aspects of the complex regulatory network underlying cellular function. Recently strengths and weaknesses we compare the methods with each other and with a reference set of previously reported protein interactions.
15,collective dynamics of smallworld networks,Networks of coupled dynamical systems have been used to model biological oscillators1 3 4 Josephson junction arrays5 6 excitable media7 neural networks8 9 10 spatial games11 genetic control networks12 and many other self-organizing systems. Ordinarily the connection topology is assumed to be either completely regular or completely random. But many biological technological and social networks lie somewhere between these two extremes. Here we explore simple models of networks that can be tuned through this middle ground: regular networks 'rewired' to introduce increasing amounts of disorder. We find that these systems can be highly clustered like regular lattices yet have small characteristic path lengths like random graphs. We call them 'small-world' networks by analogy with the small-world phenomenon13 14 (popularly known as six degrees of separation15). The neural network of the worm Caenorhabditis elegans the power grid of the western United States and the collaboration graph of film actors are shown to be small-world networks. Models of dynamical systems with small-world coupling display enhanced signal-propagation speed computational power and synchronizability. In particular infectious diseases spread more easily in small-world networks than in regular lattices.
16,network motifs simple building blocks of complex networks,"Complex networks are studied across many fields of science. {T}o uncover their structural design principles "" patterns of interconnections occurring in complex networks at numbers that are significantly higher than those in randomized networks. {W}e found such motifs in networks from biochemistry neurobiology ecology and engineering. {T}he motifs shared by ecological food webs were distinct from the motifs shared by the genetic networks of {E}scherichia coli and {S}accharomyces cerevisiae or from those found in the {W}orld {W}ide {W}eb. {S}imilar motifs were found in networks that perform information processing even though they describe elements as different as biomolecules within a cell and synaptic connections between neurons in {C}aenorhabditis elegans. {M}otifs may thus define universal classes of networks. {T}his approach may uncover the basic building blocks of most networks."""""""""
17,authoritative sources in a hyperlinked environment,"The network structure of a hyperlinked environment can be a rich source of in- formation about the content of the environment and report on experiments that demonstrate their e ectiveness in a variety of contexts on the World Wide Web. The central issue we address within our framework is the distillation of broad search topics through the discovery of \authoritative information sources on such topics. We propose and test an algorithmic formulation of the notion of authority based on the relationship between a set of relevant authoritative pages and the set of \hub pages"""""""" that join them together in the link structure. Our formulation has connections to the eigenvectors of certain matrices associated with the link graph these connections in turn motivate additional heuristics for link-based analysis. We began with the goal of discovering authoritative pages but our approach in fact identi """""
24,classes of smallworld networks,We study the statistical properties of a variety of diverse real-world networks. We present evidence of the occurrence of three classes of small-world networks: (a) scale-free networks (b) broad-scale networks characterized by a connectivity distribution that has a power law regime followed by a sharp cut

✓ Final df: (2261, 3) | saved: cleaned_articles.xlsx


In [6]:
# ============= 6) Ratings: Load & Merge Positives =============
RATINGS_CSV = "Ratings.csv"
ratings = pd.read_csv(RATINGS_CSV, low_memory=False)
ratings.columns = ['researcher_id','article_id','rating']

merged_data = ratings.merge(df, on='article_id', how='inner')
merged_data['clean_text'] = (merged_data['title'].fillna('') + ' ' + merged_data['abstract'].fillna('')).apply(preprocess_text)

positives = merged_data[merged_data['rating'] > 0].copy()

display(positives[['article_id','researcher_id','rating','clean_text']].head(5).style.hide(axis='index'))
print(f"✓ Positives merged: {positives.shape}")

article_id,researcher_id,rating,clean_text
707,4,1,formal ontolog inform system research ontolog becom increasingli widespread comput scienc commun includ knowledg engin databas design integr inform retriev extract shall use gener term inform system broadest sens collect refer applic perspect argu paper call ontolog present methodolog architectur peculiar methodolog side main peculiar adopt highli interdisciplinari approach architectur side interest aspect central role play inform system lead perspect ontolog driven inform system
710,4,1,translat approach portabl ontolog specif support share reus formal repres knowledg among ai system relat function object call ontolog paper describ mechan defin ontolog portabl represent system definit written standard format predic calculu translat system call ontolingua special represent includ frame base system well relat languag allow research share reus ontolog retain comput benefit special implement discuss translat approach portabl address sever technic problem one problem accommod stylist organiz differ among represent preserv declar content anoth translat express languag restrict languag remain system independ preserv comput effici implement system describ problem address base ontolingua ontolog domain independ represent idiom
722,4,1,ontolog silver bullet knowledg manag electron commerc ontolog develop investig time artifici intellig facilit knowledg share reus recent intellig inform integr cooper inform system inform retriev electron commerc enterpris applic integr knowledg manag broaden interest ontolog base featur provid machin process semant inform sourc commun among agent well softwar artifact human featur make ontolog backbon technolog next web gener e semant web ontolog current appli area knowledg manag larg compani wide network call center b c b g b b electron commerc nutshel ontolog enabl effect effici access heterogen distribut inform sourc given increas amount inform avail onlin kind support becom import day day author systemat introduc notion ontolog non expert reader demonstr detail appli conceptu framework improv intranet retriev corpor inform knowledg enhanc internet base electron commerc also describ ontolog languag xml rdf owl ontolog tool applic ontolog addit structur improv second edit cover recent develop relat semant web emerg web base standard languag
756,4,1,semant web entertain system belt beatl work phone rang pete answer luci line doctor offic mom need see specialist seri physic therapi session biweekli someth m go agent set appoint pete immedi agre share chauffeur
757,4,1,busi process execut languag web servic version document defin notat specifi busi process behavior base web servic notat call busi process execut languag web servic abbrevi bpel ws rest document process bpel ws export import function use web servic interfac exclus busi process describ two way execut busi process model actual behavior particip busi interact busi protocol use process descript specifi mutual visibl messag exchang behavior parti involv protocol without reveal intern behavior process descript busi protocol call abstract process bpel ws meant use model behavior execut abstract process bpel ws provid languag formal specif busi process busi interact protocol extend web servic interact model enabl support busi transact bpel ws defin interoper integr model facilit expans autom process integr intra corpor busi busi space


✓ Positives merged: (2648, 6)


In [7]:
# ============= 7) Unified TF-IDF + Profiles + Negatives + Train DF =============
df['clean_text'] = (df['title'].fillna('') + ' ' + df['abstract'].fillna('')).apply(preprocess_text)
all_articles_clean_df = df[['article_id','clean_text']].copy()

UNIFIED_TFIDF = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
UNIFIED_TFIDF.fit(all_articles_clean_df['clean_text'])

article_tfidf = UNIFIED_TFIDF.transform(all_articles_clean_df['clean_text'])

user_profiles_text = positives.groupby('researcher_id')['clean_text'].apply(' '.join).reset_index()
user_vecs = UNIFIED_TFIDF.transform(user_profiles_text['clean_text'])

all_ids = set(all_articles_clean_df['article_id'])
neg_rows = []
for i, rid in enumerate(user_profiles_text['researcher_id']):
    uvec = user_vecs[i]
    sim = cosine_similarity(uvec, article_tfidf)[0]
    sim_df = pd.DataFrame({'article_id': all_articles_clean_df['article_id'], 'similarity': sim})
    liked = set(positives[positives['researcher_id']==rid]['article_id'])
    rated_any = set(merged_data[merged_data['researcher_id']==rid]['article_id'])
    unrated = all_ids - rated_any
    unrated_sim = sim_df[sim_df['article_id'].isin(unrated)].sort_values('similarity', ascending=True)
    k = len(liked)
    neg_pick = unrated_sim.head(k)
    for aid in neg_pick['article_id']:
        neg_rows.append({'researcher_id': rid, 'article_id': aid, 'rating': 0})

negative_samples_df = pd.DataFrame(neg_rows).merge(all_articles_clean_df, on='article_id', how='left')

positive_cols = ['researcher_id','article_id','rating','clean_text']
final_training_df = pd.concat([positives[positive_cols], negative_samples_df], ignore_index=True)
final_training_df = final_training_df.sample(frac=1, random_state=42).reset_index(drop=True)
final_training_df.to_csv("similarity_sampled_training_data.csv", index=False)

display(final_training_df.head(8).style.hide(axis='index'))
print(f"✓ Training data ready: {final_training_df.shape}")

researcher_id,article_id,rating,clean_text
89,4224,1,biodivers datadivers biodivers data intens scienc layer valu emerg infrastructur argu layer process rel irrevers oper simultan concret level field databas abstract one code relationship disciplin product gener ontolog final maintain scienc studi disciplin abl make signific contribut design robust flexibl databas recogn perform charact infrastructur introduct form scientif work studi sociologist scienc lead laboratori scientif pap
204,778,1,global map yeast genet interact network genet interact network contain approxim gene approxim interact map cross mutat differ queri gene set approxim viabl gene yeast delet mutant score doubl mutant progeni fit defect network connect predict function interact often occur among function relat gene therefor posit gene partial map network predict genet interact digen interact common yeast similar network may underli complex genet associ inherit phenotyp organ
158,10044,1,structur evolut onlin social network work one friend relat within flickr photo share applic yahoo social network network togeth compris excess million peopl million friendship link annot metadata captur time everi event life network show network may segment three region singleton particip network isol commun overwhelmingli display star structur giant compon anchor well connect core region persist even absenc star give detail character structur evolut region also present simpl model network growth captur aspect compon structur model follow experiment result character user either passiv member network invit encourag offlin friend acquaint migrat onlin linker fulli particip social evolut network show simpl model two numer paramet abl produc synthet network accur reflect structur real world network
283,15628,0,fulli homomorph encrypt use ideal lattic propos fulli homomorph encrypt scheme e provid gener result construct encrypt scheme permit evalu arbitrari circuit suffic construct encrypt scheme evalu slightli augment version decrypt circuit call scheme evalu augment decrypt circuit bootstrapp
442,7448,0,classbas ngram model natur languag address problem predict word previou word sampl text particular depend natur underli statist
354,16778,0,deriv consequ genom variant ensembl api snp effect predictor summari tool predict effect newli discov genom variant known transcript indispens priorit categor variant ensembl nan
271,7793,1,crosslay design survey road ahead late approach cross layer design via interpret impli present sever cross layer design propos propos involv differ layer protocol stack address cellular ad hoc network also work relat implement cross layer interact high time variou individu effort put perspect holist view taken articl take step direct present survey literatur area cross layer design take stock ongo work suggest definit cross layer design discuss basic type cross layer design exampl drawn literatur categor initi propos cross layer interact may implement highlight open challeng new opportun cross layer design design present cross layer design propos start address move ahead
442,1538,1,creat gene ontolog resourc design implement exponenti growth volum access biolog inform gener confus voic surround annot molecular inform gene product gene ontolog go project seek provid set structur vocabulari specif biolog domain use describ gene product organ work includ build three extens ontolog describ molecular function cellular compon provid commun databas resourc support use ontolog go consortium initi scientist associ three model organ databas sgd saccharomyc genom databas flybas drosophila genom databas mgd gxd mous genom informat databas addit model organ databas group join project model organ inform system annot gene gene product use go vocabulari term incorpor annot respect model organ databas databas contribut annot file share go data resourc access public http www geneontolog org go site use commun recov go vocabulari access annot gene product data set model orga

✓ Training data ready: (5296, 4)


In [8]:
# ============= 8) Train/Test Split =============
data = pd.read_csv("similarity_sampled_training_data.csv")
data['clean_text'] = data['clean_text'].fillna('')

X, y = data['clean_text'], data['rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"✓ Split: Train={len(X_train)}, Test={len(X_test)}")

✓ Split: Train=4236, Test=1060


In [9]:
# ============= 9) Train Models (Unified TF-IDF) + Evaluate =============
train_mat = UNIFIED_TFIDF.transform(X_train)
test_mat  = UNIFIED_TFIDF.transform(X_test)

lr_model = LogisticRegression(random_state=42, solver='liblinear')
nb_model = MultinomialNB()

lr_model.fit(train_mat, y_train)
nb_model.fit(train_mat, y_train)

y_pred_lr = lr_model.predict(test_mat)
y_pred_nb = nb_model.predict(test_mat)

def compact_report(y_true, y_pred, title):
    rep = classification_report(y_true, y_pred, output_dict=True)
    return {
        'model': title,
        'accuracy': rep['accuracy'],
        'f1_macro': rep['macro avg']['f1-score'],
        'precision_macro': rep['macro avg']['precision'],
        'recall_macro': rep['macro avg']['recall'],
    }

eval_compare = pd.DataFrame([
    compact_report(y_test, y_pred_lr, 'LogisticRegression (Unified TF-IDF)'),
    compact_report(y_test, y_pred_nb, 'MultinomialNB (Unified TF-IDF)')
])[['model','accuracy','f1_macro','precision_macro','recall_macro']]

display(eval_compare.style.hide(axis='index').format({
    'accuracy':'{:.4f}','f1_macro':'{:.4f}','precision_macro':'{:.4f}','recall_macro':'{:.4f}'
}))

eval_compare.to_csv("Eval_LR_vs_NB_UnifiedTFIDF.csv", index=False)
print("✓ Saved: Eval_LR_vs_NB_UnifiedTFIDF.csv")

model,accuracy,f1_macro,precision_macro,recall_macro
LogisticRegression (Unified TF-IDF),0.9038,0.9036,0.9059,0.9038
MultinomialNB (Unified TF-IDF),0.8896,0.8893,0.8945,0.8896


✓ Saved: Eval_LR_vs_NB_UnifiedTFIDF.csv


In [10]:
# ============= 10) Helpers: Mapping + Explainability =============
all_articles_map = pd.merge(df[['article_id','title']],
                            df[['article_id','clean_text']], on='article_id', how='left')

def explain_lr_top_words(article_clean_text, n=5):
    vocab = UNIFIED_TFIDF.get_feature_names_out()
    coef = lr_model.coef_[0]
    art_vec = UNIFIED_TFIDF.transform([article_clean_text])
    idx, vals = art_vec.indices, art_vec.data
    contrib = coef[idx] * vals
    out = pd.DataFrame({'word':[vocab[i] for i in idx],
                        'coef':coef[idx],'tfidf':vals,'contribution':contrib})
    out = out[out['contribution'] > 0].sort_values('contribution', ascending=False)
    return out.head(n).reset_index(drop=True)

def explain_nb_top_words(article_clean_text, n=5):
    vocab = UNIFIED_TFIDF.get_feature_names_out()
    logp = nb_model.feature_log_prob_[1]
    art_vec = UNIFIED_TFIDF.transform([article_clean_text])
    idx = art_vec.indices
    out = pd.DataFrame({'word':[vocab[i] for i in idx],
                        'log_prob_liked':logp[idx]}).sort_values('log_prob_liked', ascending=False)
    return out.head(n).reset_index(drop=True)

print("✓ Explain helpers ready")

✓ Explain helpers ready


In [20]:
# ============= 11) Hybrid Recommender (Unified) + Demo + Export =============
user_id_to_index    = pd.Series(user_profiles_text.index, index=user_profiles_text['researcher_id']).to_dict()
article_id_to_index = pd.Series(all_articles_clean_df.index, index=all_articles_clean_df['article_id']).to_dict()
all_profile_scores  = cosine_similarity(user_vecs, article_tfidf)

def get_hybrid_recommendations(researcher_id, all_map_df, ratings_df,
                               content_weight=0.4, profile_weight=0.6, n=5):
    rated = set(ratings_df[ratings_df['researcher_id'] == researcher_id]['article_id'])
    cands = all_map_df[~all_map_df['article_id'].isin(rated)].copy()
    if cands.empty:
        return pd.DataFrame(columns=['article_id','title','score','score_content','score_profile'])
    cand_mat = UNIFIED_TFIDF.transform(cands['clean_text'])
    cands['score_content'] = lr_model.predict_proba(cand_mat)[:,1]
    uid = user_id_to_index.get(researcher_id, None)
    if uid is None:
        cands['score_profile'] = 0.0
    else:
        idxs = cands['article_id'].map(article_id_to_index).values
        cands['score_profile'] = all_profile_scores[uid][idxs]
    cands['score'] = content_weight*cands['score_content'] + profile_weight*cands['score_profile']
    return cands.sort_values('score', ascending=False).head(n)[['article_id','title','score','score_content','score_profile']]

def recommend(researcher_id, n=5):
    return get_hybrid_recommendations(researcher_id, all_articles_map, ratings, n=n)

sample_user = int(ratings['researcher_id'].mode().iloc[0])
demo = recommend(sample_user, n=5)
display(demo.style.hide(axis='index').format({'score':'{:.4f}','score_content':'{:.4f}','score_profile':'{:.4f}'}))

if not demo.empty:
    aid = int(demo.iloc[0]['article_id'])
    atxt = all_articles_map.loc[all_articles_map['article_id']==aid, 'clean_text'].iloc[0]
    print("\nLR Explanation (Top +contribution words):")
    display(explain_lr_top_words(atxt, n=5).style.hide(axis='index').format({'coef':'{:.4f}','tfidf':'{:.4f}','contribution':'{:.4f}'}))
    print("\nNB Explanation (Top log-prob words for liked):")
    display(explain_nb_top_words(atxt, n=5).style.hide(axis='index').format({'log_prob_liked':'{:.4f}'}))

rows = []
for rid in ratings['researcher_id'].dropna().unique():
    r = recommend(int(rid), n=5)
    for _, x in r.iterrows():
        rows.append({'researcher_id': int(rid),
                     'article_id': int(x['article_id']),
                     'title': x['title'],
                     'score': float(x['score']),
                     'score_content': float(x['score_content']),
                     'score_profile': float(x['score_profile'])})
hybrid_out = pd.DataFrame(rows)
hybrid_out.to_csv("Top5_Recommendations_ALL_HYBRID.csv", index=False)
print("✓ Saved: Top5_Recommendations_ALL_HYBRID.csv | shape=", hybrid_out.shape)

article_id,title,score,score_content,score_profile
2490,evaluating collaborative filtering recommender systems,0.5949,0.9364,0.3672
2789,empirical analysis of predictive algorithms for collaborative filtering,0.5810,0.9734,0.3195
11204,google news personalization scalable online collaborative filtering,0.5797,0.9169,0.3550
8908,web page recommender system based on folksonomy mining for itng submissions,0.5772,0.8948,0.3655
11769,a contentcollaborative recommender that exploits wordnetbased user profiles for neighborhood formation,0.5760,0.9506,0.3263



LR Explanation (Top +contribution words):


word,coef,tfidf,contribution
system,2.1792,0.1336,0.2911
evalu,0.6599,0.3371,0.2224
recommend,1.0826,0.1984,0.2148
use,3.2686,0.0494,0.1614
metric,0.5024,0.2863,0.1438



NB Explanation (Top log-prob words for liked):


word,log_prob_liked
use,-6.7157
system,-6.7462
user,-7.1477
research,-7.2738
present,-7.2852


✓ Saved: Top5_Recommendations_ALL_HYBRID.csv | shape= (1000, 6)


In [12]:
# ============= 12) Diversity: LR vs NB (content-only) + Hybrid =============
from sklearn.metrics.pairwise import cosine_similarity as cos

def top5_content(model):
    rows = []
    for rid in ratings['researcher_id'].dropna().unique():
        rated = set(ratings[ratings['researcher_id']==rid]['article_id'])
        cands = all_articles_map[~all_articles_map['article_id'].isin(rated)].copy()
        if cands.empty: continue
        Xc = UNIFIED_TFIDF.transform(cands['clean_text'])
        sc = model.predict_proba(Xc)[:,1]
        cands['score'] = sc
        topn = cands.sort_values('score', ascending=False).head(5)
        for _, r in topn.iterrows():
            rows.append({'researcher_id': int(rid),'article_id': int(r['article_id']),'title': r['title'],'score': float(r['score'])})
    return pd.DataFrame(rows)

def intra_list_similarity_ids(ids):
    if len(ids) < 2: return np.nan
    texts = all_articles_map[all_articles_map['article_id'].isin(ids)]['clean_text']
    M = UNIFIED_TFIDF.transform(texts)
    S = cos(M)
    iu = np.triu_indices_from(S, k=1)
    return float(np.mean(S[iu]))

def avg_diversity(df_top5):
    sims = []
    for rid, g in df_top5.groupby('researcher_id'):
        ids = list(g.sort_values('score', ascending=False)['article_id'])
        sims.append(intra_list_similarity_ids(ids))
    avg_sim = np.nanmean(sims) if sims else np.nan
    return avg_sim, (None if pd.isna(avg_sim) else 1-avg_sim)

top5_lr_content = top5_content(lr_model)
top5_nb_content = top5_content(nb_model)

avg_sim_lr, div_lr = avg_diversity(top5_lr_content)
avg_sim_nb, div_nb = avg_diversity(top5_nb_content)
avg_sim_h, div_h  = avg_diversity(hybrid_out.rename(columns={'score':'score'}))

div_compare = pd.DataFrame({
    'model':['LR Content','NB Content','Hybrid'],
    'avg_intra_list_similarity':[avg_sim_lr, avg_sim_nb, avg_sim_h],
    'diversity_score (1 - sim)':[div_lr, div_nb, div_h]
})
display(div_compare.style.hide(axis='index').format({'avg_intra_list_similarity':'{:.4f}','diversity_score (1 - sim)':'{:.4f}'}))
div_compare.to_csv("Diversity_LR_vs_NB_vs_Hybrid.csv", index=False)
print("✓ Saved: Diversity_LR_vs_NB_vs_Hybrid.csv")

model,avg_intra_list_similarity,diversity_score (1 - sim)
LR Content,0.1670,0.8330
NB Content,0.0883,0.9117
Hybrid,0.1693,0.8307


✓ Saved: Diversity_LR_vs_NB_vs_Hybrid.csv


In [13]:
# ============= 13) Cold-Start =============
cold_user = int(ratings['researcher_id'].iloc[18])
rated_cold = set(ratings[ratings['researcher_id']==cold_user]['article_id'])
cands = all_articles_map[~all_articles_map['article_id'].isin(rated_cold)].copy()
Xc = UNIFIED_TFIDF.transform(cands['clean_text'])
cands['score'] = lr_model.predict_proba(Xc)[:,1]
cold_top5 = cands.sort_values('score', ascending=False).head(5)[['article_id','title','score']]
display(cold_top5.style.hide(axis='index').format({'score':'{:.4f}'}))
cold_top5.to_csv("Top5_ColdStart_ContentOnly.csv", index=False)
print("✓ Saved: Top5_ColdStart_ContentOnly.csv")

article_id,title,score
4649,a blocksorting lossless data compression algorithm,0.9837
1930,software architecture in practice,0.9800
5268,introduction to information retrieval,0.9792
10661,composing specifications,0.9791
1137,software architecture perspectives on an emerging discipline,0.9790


✓ Saved: Top5_ColdStart_ContentOnly.csv


In [19]:
# ============= 14) Word2Vec Features + LR Compare =============
!pip install gensim
from gensim.models import Word2Vec

sentences = df['clean_text'].apply(lambda s: str(s).split()).tolist()
w2v = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=2, workers=4, sg=0)

def doc_vector(text, model, dim=100):
    ws = str(text).split()
    vecs = [model.wv[w] for w in ws if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

X_train_w2v = np.vstack([doc_vector(t, w2v) for t in X_train])
X_test_w2v  = np.vstack([doc_vector(t, w2v) for t in X_test])

lr_w2v = LogisticRegression(random_state=42, solver='liblinear')
lr_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = lr_w2v.predict(X_test_w2v)

compare_feats = pd.DataFrame([
    {'features':'Unified TF-IDF (1–2-grams, 20k)','model':'LR','accuracy':classification_report(y_test, y_pred_lr, output_dict=True)['accuracy']},
    {'features':'Word2Vec (avg 100-dim)','model':'LR','accuracy':classification_report(y_test, y_pred_w2v, output_dict=True)['accuracy']}
])[['features','model','accuracy']]

display(compare_feats.style.hide(axis='index').format({'accuracy':'{:.4f}'}))
compare_feats.to_csv("Compare_Features_LR_TFIDF_vs_W2V.csv", index=False)
print("✓ Saved: Compare_Features_LR_TFIDF_vs_W2V.csv")

features,model,accuracy
"Unified TF-IDF (1–2-grams, 20k)",LR,0.9038
Word2Vec (avg 100-dim),LR,0.7538


✓ Saved: Compare_Features_LR_TFIDF_vs_W2V.csv


In [16]:
# ============= 15) Final Delivery Bundle (Paths) =============
bundle = pd.DataFrame({
    'file': [
        'cleaned_articles.xlsx',
        'similarity_sampled_training_data.csv',
        'Eval_LR_vs_NB_UnifiedTFIDF.csv',
        'Top5_Recommendations_ALL_HYBRID.csv',
        'Top5_ColdStart_ContentOnly.csv',
        'Diversity_LR_vs_NB_vs_Hybrid.csv',
        'Compare_Features_LR_TFIDF_vs_W2V.csv'
    ]
})
display(bundle)

,file
0,cleaned_articles.xlsx
1,similarity_sampled_training_data.csv
2,Eval_LR_vs_NB_UnifiedTFIDF.csv
3,Top5_Recommendations_ALL_HYBRID.csv
4,Top5_ColdStart_ContentOnly.csv
5,Diversity_LR_vs_NB_vs_Hybrid.csv
6,Compare_Features_LR_TFIDF_vs_W2V.csv
